# TP 4 — Construire des variables pour la fraude
## Séance 4 : Feature engineering sur l’historique des clients
### Version corrigée (enseignant)

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

La banque vous confie l’historique de **14 978 transactions** de **300 clients** (1er janvier au 1er mars 2026). Environ 1 % sont frauduleuses. Le montant seul ne suffisant pas (TP 3), vous allez construire des variables qui comparent chaque transaction à **l’historique du client** :

1. variables temporelles et de contexte ;
2. rapport entre le montant et le montant habituel du client ;
3. vélocité : nombre de transactions récentes du client, temps depuis la précédente ;
4. nouveauté : pays ou catégorie jamais utilisés par le client ;
5. vérification que les variables **n’utilisent pas le futur** ;
6. mesure de l’utilité de chaque variable ;
7. comparaison de règles de détection avec et sans ces variables.

**Règle d’or** : pour chaque transaction, une variable ne doit dépendre que des transactions **précédentes** du même client.

## Étape 1 — Données et fonction d’évaluation (fournies)

Exécutez la cellule suivante **sans la modifier**. Elle génère le jeu de données et définit `evaluer(y, alerte)`, qui affiche la matrice de confusion, la précision et le rappel.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generer_transactions_clients(seed=11):
    rng = np.random.default_rng(seed)
    n_clients = 300
    jours = 60
    debut = pd.Timestamp('2026-01-01')

    categories = ['alimentation', 'restauration', 'transport', 'vetements', 'electronique', 'voyage', 'jeux_en_ligne']
    p_cat = [0.30, 0.22, 0.15, 0.12, 0.08, 0.06, 0.07]
    decalage = {'alimentation': 0.0, 'restauration': -0.1, 'transport': -0.4, 'vetements': 0.5,
                'electronique': 1.2, 'voyage': 1.5, 'jeux_en_ligne': -0.3}

    # Profils clients : pays habituel, niveau de dépenses, heure habituelle, nombre de transactions
    clients = pd.DataFrame({
        'id_client': [f'C{i:03d}' for i in range(n_clients)],
        'pays_hab': rng.choice(['FR', 'ES', 'IT', 'DE'], size=n_clients, p=[0.9, 0.04, 0.03, 0.03]),
        'mu': rng.normal(3.3, 0.35, n_clients),
        'heure_pref': rng.choice([12, 17, 19], size=n_clients),
        'n_tx': rng.poisson(48, n_clients),
    })

    lignes = []
    for c in clients.itertuples():
        k = c.n_tx
        cat = rng.choice(categories, size=k, p=p_cat)
        montant = np.array([rng.lognormal(c.mu + decalage[x], 0.4) for x in cat])
        jour = rng.integers(0, jours, k)
        heure = np.clip(np.round(rng.normal(c.heure_pref, 3, k)), 7, 23).astype(int)
        nuit = rng.random(k) < 0.04      # quelques achats nocturnes légitimes
        heure[nuit] = rng.integers(0, 6, nuit.sum())
        minute = rng.integers(0, 60, k)
        t = debut + pd.to_timedelta(jour, unit='D') + pd.to_timedelta(heure, unit='h') + pd.to_timedelta(minute, unit='m')
        canal = rng.choice(['carte_presente', 'sans_contact', 'en_ligne'], size=k, p=[0.45, 0.30, 0.25])
        canal[cat == 'jeux_en_ligne'] = 'en_ligne'
        pays = np.where(rng.random(k) < 0.95, c.pays_hab, rng.choice(['FR', 'ES', 'IT', 'DE', 'US'], size=k))
        for i in range(k):
            lignes.append((c.id_client, t[i], round(montant[i], 2), cat[i], canal[i], pays[i], 0))
            # achats groupés légitimes : une 2e transaction peu après (5 % des cas)
            if rng.random() < 0.05 and canal[i] != 'en_ligne':
                lignes.append((c.id_client, t[i] + pd.Timedelta(minutes=int(rng.integers(5, 40))),
                               round(rng.lognormal(c.mu - 0.2, 0.4), 2), cat[i], canal[i], pays[i], 0))
    df = pd.DataFrame(lignes, columns=['id_client', 'date_heure', 'montant', 'categorie', 'canal', 'pays', 'fraude'])

    # Fraudes de type A : rafales sur la carte d'un client (anomalie collective)
    victimes_a = rng.choice(n_clients, size=22, replace=False)
    fraudes = []
    for v in victimes_a:
        cl = clients.iloc[v]
        nb = int(rng.integers(4, 8))
        jour = int(rng.integers(5, jours))
        heure = int(rng.integers(0, 6)) if rng.random() < 0.7 else int(rng.integers(6, 24))
        t0 = debut + pd.Timedelta(days=jour, hours=heure, minutes=int(rng.integers(0, 60)))
        etranger = rng.random() < 0.7
        pays_f = rng.choice(['US', 'NG', 'DE', 'ES']) if etranger else cl['pays_hab']
        for j in range(nb):
            cat = rng.choice(['electronique', 'jeux_en_ligne', 'voyage', 'vetements'], p=[0.35, 0.3, 0.15, 0.2])
            fraudes.append((cl['id_client'], t0 + pd.Timedelta(minutes=int(j * rng.integers(2, 6))),
                            round(rng.lognormal(4.0, 0.5), 2), cat, 'en_ligne', pays_f, 1))
    # Fraudes de type B : transaction isolée très inhabituelle pour le client
    victimes_b = rng.choice(n_clients, size=40, replace=False)
    for v in victimes_b:
        cl = clients.iloc[v]
        t0 = debut + pd.Timedelta(days=int(rng.integers(5, jours)), hours=int(rng.integers(0, 24)), minutes=int(rng.integers(0, 60)))
        pays_f = rng.choice([p for p in ['US', 'NG', 'DE', 'ES', 'IT'] if p != cl['pays_hab']])
        fraudes.append((cl['id_client'], t0, round(float(np.exp(cl['mu'] + rng.normal(1.8, 0.3))), 2),
                        rng.choice(['electronique', 'voyage']), 'en_ligne' if rng.random() < 0.8 else 'carte_presente', pays_f, 1))
    df = pd.concat([df, pd.DataFrame(fraudes, columns=df.columns)], ignore_index=True)

    df = df.sort_values('date_heure', kind='stable').reset_index(drop=True)
    df.insert(0, 'id_transaction', [f'T{i:05d}' for i in range(len(df))])
    return df

df = generer_transactions_clients()

def evaluer(y, alerte, afficher=True):
    alerte = np.asarray(alerte).astype(int)
    y = np.asarray(y)
    tp = int(((alerte == 1) & (y == 1)).sum())
    fp = int(((alerte == 1) & (y == 0)).sum())
    fn = int(((alerte == 0) & (y == 1)).sum())
    tn = int(((alerte == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    rappel = tp / (tp + fn)
    if afficher:
        print(f'Alertes={tp + fp}  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
        print(f'Précision={precision:.2f}  Rappel={rappel:.2f}')
    return precision, rappel

print(df.shape, '|', df['id_client'].nunique(), 'clients |', df['fraude'].sum(), 'fraudes')
df.head()

## Étape 2 — Tri et variables temporelles

Les variables d’historique exigent que les transactions soient triées **par client, puis par date**. Complétez la cellule (remplacez chaque `...`).

In [ ]:
df = df.sort_values(['id_client', 'date_heure']).reset_index(drop=True)

df['heure'] = df['date_heure'].dt.hour
df['jour_semaine'] = df['date_heure'].dt.dayofweek        # 0 = lundi ... 6 = dimanche
df['weekend'] = (df['jour_semaine'] >= 5).astype(int)     # 1 si samedi ou dimanche
df['nuit'] = (df['heure'] < 6).astype(int)                # 1 si heure < 6
df['etranger'] = (df['pays'] != 'FR').astype(int)         # 1 si pays différent de 'FR'
df['en_ligne'] = (df['canal'] == 'en_ligne').astype(int)  # 1 si canal en ligne

## Étape 3 — Le montant par rapport à l’habitude du client

On compare le montant de chaque transaction à la **moyenne des montants précédents** du même client. Attention : la transaction courante ne doit pas entrer dans sa propre moyenne.

In [ ]:
g = df.groupby('id_client')

df['rang_client'] = g.cumcount()                                                  # 0 pour la première transaction
df['moy_hist'] = g['montant'].transform(lambda s: s.shift().expanding().mean())  # moyenne des montants PRÉCÉDENTS
df['ratio_montant'] = (df['montant'] / df['moy_hist']).fillna(1.0)               # 1.0 sans historique
df[['id_client', 'montant', 'rang_client', 'moy_hist', 'ratio_montant']].head(6)

## Étape 4 — Vélocité : fréquence des transactions

On compte, pour chaque transaction, le nombre de transactions **précédentes** du client dans les 10 dernières minutes, la dernière heure et les dernières 24 heures.

Indice : `df.groupby('id_client').rolling(fenetre, on='date_heure', closed='left')['montant'].count()` donne le comptage ; ajoutez `.fillna(0).to_numpy()` pour l’insérer dans le DataFrame (il est trié par client puis par date).

In [ ]:
for fenetre, nom in [('10min', 'nb_10min'), ('1h', 'nb_1h'), ('24h', 'nb_24h')]:
    df[nom] = (df.groupby('id_client')
                 .rolling(fenetre, on='date_heure', closed='left')['montant']
                 .count().fillna(0).to_numpy())

# Temps écoulé depuis la transaction précédente du client, en minutes
# (10080 = une semaine quand il n'y a pas de transaction précédente)
df['minutes_depuis_derniere'] = (g['date_heure'].diff().dt.total_seconds() / 60).fillna(10080)
df[['id_client', 'date_heure', 'nb_10min', 'nb_1h', 'nb_24h', 'minutes_depuis_derniere']].head(8)

## Étape 5 — Nouveauté : pays et catégories jamais utilisés

Une transaction est « nouvelle » si c’est la **première fois** que le client utilise ce pays (ou cette catégorie), et si ce n’est pas sa toute première transaction.

In [ ]:
# Première apparition du pays / de la catégorie pour ce client
df['pays_nouveau'] = ((df.groupby(['id_client', 'pays']).cumcount() == 0) & (df['rang_client'] > 0)).astype(int)
df['cat_nouvelle'] = ((df.groupby(['id_client', 'categorie']).cumcount() == 0) & (df['rang_client'] > 0)).astype(int)
print(df['pays_nouveau'].sum(), 'transactions dans un pays nouveau pour le client')

## Étape 6 — Vérification : aucune variable n’utilise le futur

Regardez les variables de la **première transaction** de quelques clients. Elles ne peuvent dépendre d’aucune transaction passée, puisqu’il n’y en a pas.

In [ ]:
# Variables de la première transaction de 3 clients
df[df['rang_client'] == 0][['id_client', 'moy_hist', 'ratio_montant', 'nb_10min', 'nb_1h', 'nb_24h',
                            'minutes_depuis_derniere', 'pays_nouveau']].head(3)

## Étape 7 — Quelles variables séparent la fraude ?

Calculez le taux de fraude (en %) par modalité ou par tranche de chaque variable.

**Variables binaires**

In [ ]:
# Taux de fraude (%) selon chaque variable binaire : 0 ou 1
for col in ['nuit', 'weekend', 'etranger', 'en_ligne', 'pays_nouveau', 'cat_nouvelle']:
    print(col)
    print((df.groupby(col)['fraude'].agg(['mean', 'sum', 'count']).assign(mean=lambda t: (t['mean'] * 100).round(2))))

**Vélocité**

In [ ]:
# Taux de fraude (%) selon le nombre de transactions des 10 dernières minutes (0, 1, 2, 3 et plus)
(df.groupby(df['nb_10min'].clip(upper=3))['fraude'].agg(['mean', 'sum', 'count'])
   .assign(mean=lambda t: (t['mean'] * 100).round(2)))

**Rapport au montant habituel**

In [ ]:
# Taux de fraude (%) selon le rapport au montant habituel
tranches = pd.cut(df['ratio_montant'], [0, 1, 2, 5, np.inf], right=False)
(df.groupby(tranches, observed=True)['fraude'].agg(['mean', 'sum', 'count'])
   .assign(mean=lambda t: (t['mean'] * 100).round(2)))

In [ ]:
# Graphique en barres : taux de fraude selon nb_10min (plafonné à 3)
taux = df.groupby(df['nb_10min'].clip(upper=3))['fraude'].mean() * 100
plt.figure(figsize=(6, 3.5))
plt.bar(taux.index.astype(int).astype(str), taux.values)
plt.xlabel('Transactions dans les 10 dernières minutes')
plt.ylabel('Taux de fraude (%)')
plt.show()

## Étape 8 — Des règles de détection avec les nouvelles variables

Comparez quatre règles :

- **Règle 1** (point de départ) : montant supérieur à 150 € ;
- **Règle 2** : au moins 2 transactions du client dans les 10 dernières minutes ;
- **Règle 3** : règle 2, **ou** (rapport de montant > 3 **et** pays nouveau) ;
- **Règle 4** : règle 3, **ou** (nuit **et** pays étranger).

In [ ]:
regle1 = df['montant'] > 150
regle2 = df['nb_10min'] >= 2
regle3 = regle2 | ((df['ratio_montant'] > 3) & (df['pays_nouveau'] == 1))
regle4 = regle3 | ((df['nuit'] == 1) & (df['etranger'] == 1))

for nom, regle in [('Règle 1', regle1), ('Règle 2', regle2), ('Règle 3', regle3), ('Règle 4', regle4)]:
    print('---', nom, '---')
    evaluer(df['fraude'], regle)

## Questions

Répondez dans les cellules de texte situées sous chaque question.

**Question 1.** Une transaction de 90 € est-elle suspecte ? Quelle variable construite dans ce TP permet de répondre à cette question pour un client donné ? Pourquoi le montant seul ne suffit-il pas ?

**Réponse 1.** Cela dépend du client. Pour un client qui dépense habituellement environ 20 €, 90 € est inhabituel ; pour un client qui dépense environ 80 €, c’est banal. La variable `ratio_montant` (montant divisé par la moyenne des montants précédents du client) traduit cela : elle vaut environ 4,5 dans le premier cas et 1,1 dans le second. Le montant seul ne suffit pas parce qu’il ignore ce qui est normal **pour ce client**.

**Question 2.** Que valent `moy_hist`, `ratio_montant`, `nb_10min`, `nb_1h` et `minutes_depuis_derniere` pour la première transaction d’un client ? Pourquoi ces valeurs prouvent-elles que les variables n’utilisent pas le futur ?

**Réponse 2.** Pour la première transaction d’un client : `moy_hist` est manquante (NaN, pas d’historique), `ratio_montant` vaut 1,0 (valeur neutre), `nb_10min` et `nb_1h` valent 0, et `minutes_depuis_derniere` vaut 10 080 (valeur par convention : une semaine). Ces valeurs sont cohérentes : il n’y a **aucune** transaction passée, donc aucune information n’est utilisée. Si les variables regardaient le futur, la première transaction aurait déjà des valeurs calculées à partir des suivantes.

**Question 3.** Comment évolue le taux de fraude avec le nombre de transactions des 10 dernières minutes ? De quel type d’anomalie (ponctuelle, contextuelle, collective) s’agit-il ?

**Réponse 3.**

| Transactions dans les 10 dernières minutes | Taux de fraude |
|---|---|
| 0 | 0,44 % (64 sur 14 607) |
| 1 | 9,8 % (30 sur 306) |
| 2 | 97,2 % (35 sur 36) |
| 3 et plus | 100 % (29 sur 29) |

Le taux de fraude explose dès qu’il y a 2 transactions récentes : c’est la signature d’une **rafale** (par exemple un test de carte volée). Il s’agit d’une **anomalie collective** : chaque transaction prise seule peut sembler normale, c’est leur enchaînement qui est anormal.

**Question 4.** Parmi `nuit`, `weekend`, `etranger`, `en_ligne`, `pays_nouveau` et `cat_nouvelle`, lesquelles sont informatives ? Laquelle ne l’est pas ? Que faites-vous de cette dernière ?

**Réponse 4.**

| Variable | Taux si 0 | Taux si 1 |
|---|---|---|
| nuit | 0,44 % | 14,2 % |
| pays_nouveau | 0,81 % | 8,2 % |
| etranger | 0,34 % | 5,7 % |
| en_ligne | 0,05 % | 3,4 % |
| cat_nouvelle | 0,97 % | 1,7 % |
| weekend | 1,05 % | 1,06 % |

`nuit`, `pays_nouveau` et `etranger` sont très informatives, `en_ligne` aussi (153 fraudes sur 158 sont en ligne), `cat_nouvelle` l’est peu. **`weekend` n’apporte rien** : le taux est identique (environ 1,05 %) que la transaction ait lieu en semaine ou le week-end. On peut l’écarter : une variable inutile ajoute du bruit et complique l’explication. Le rapport au montant habituel est aussi informatif : le taux passe de 0,3 % (rapport < 1) à 9,2 % (rapport ≥ 5).

**Question 5.** Comparez les quatre règles (nombre d’alertes, précision, rappel). Que gagne-t-on et que perd-on à chaque ajout ? Pourquoi la règle 2 ne détecte-t-elle jamais la première transaction d’une rafale ?

**Réponse 5.**

| Règle | Alertes | Précision | Rappel |
|---|---|---|---|
| 1. montant > 150 € | 504 | 0,06 | 0,20 |
| 2. nb_10min ≥ 2 | 65 | 0,98 | 0,41 |
| 3. règle 2 ou (ratio > 3 et pays nouveau) | 112 | 0,84 | 0,59 |
| 4. règle 3 ou (nuit et étranger) | 209 | 0,54 | 0,72 |

La règle 1 (montant seul) est très mauvaise : beaucoup de fausses alertes et peu de fraudes détectées. La règle 2, qui utilise la vélocité, est presque parfaite en précision (0,98) mais ne détecte que 41 % des fraudes. Chaque ajout de condition **augmente le rappel** (0,41, puis 0,59, puis 0,72) mais **fait baisser la précision** (0,98, 0,84, 0,54) : c’est le compromis habituel. La règle 2 ne détecte jamais la première transaction d’une rafale parce qu’à cet instant il n’y a encore aucune transaction récente : `nb_10min` ne compte que le **passé**. La détection commence à la deuxième transaction.

**Question 6.** Pourquoi faut-il `shift()` et `closed='left'` ? Que se passerait-il si on utilisait la moyenne du client sur toute la période ? Citez deux difficultés pour calculer ces variables en production.

**Réponse 6.** `shift()` décale la série d’une ligne : sans lui, la moyenne contiendrait le montant de la transaction courante. `closed='left'` exclut la transaction courante de sa propre fenêtre : sans lui, `nb_10min` vaudrait au moins 1 pour toutes les transactions. Avec la moyenne du client sur toute la période, chaque transaction serait comparée à une moyenne qui inclut des transactions **futures** (et elle-même) : en production, ces informations n’existent pas encore, donc le modèle paraîtrait bon en test et s’effondrerait en production (fuite de données).

Difficultés en production : calculer les fenêtres glissantes **en quelques millisecondes** (il faut conserver l’historique récent de chaque client et le mettre à jour à chaque transaction) ; gérer les clients sans historique (démarrage à froid) ; garantir que les variables calculées en temps réel sont **identiques** à celles utilisées à l’entraînement.

## Pour aller plus loin (facultatif)

1. Calculez `moy_client_globale = g['montant'].transform('mean')` (moyenne sur toute la période, futur compris) et un rapport `ratio_fuite`. Pourquoi ne peut-on pas utiliser cette variable en production ?
2. Remplacez `nb_10min` par `nb_1h` dans la règle 2. Que deviennent la précision et le rappel ?
3. Construisez une variable `ecart_heure` : écart entre l’heure de la transaction et l’heure moyenne **précédente** du client.

## Pistes pour « Pour aller plus loin »

- `moy_client_globale` utilise des transactions postérieures (et la transaction courante) : en production, au moment de la décision, ces informations n’existent pas. Une variable de ce type améliore artificiellement les résultats en test : c’est une fuite de données.
- Avec `nb_1h >= 2`, on détecte plus de fraudes (rappel d’environ 0,47) mais la précision tombe à environ 0,43, car les achats groupés légitimes (plusieurs achats en magasin dans l’heure) déclenchent des alertes. La fenêtre de 10 minutes est plus sélective.
- Pour `ecart_heure`, utilisez le même principe que `moy_hist` : `shift().expanding().mean()` sur l’heure, puis la différence avec l’heure de la transaction (en tenant compte du fait que 23 h et 1 h sont proches).